### Code funzionante

LSB --- MSB in OpenFHE

In [2127]:
import math
import numpy as np
import matplotlib.pyplot as plt
from random import randrange
import random

In [2128]:
bits = 64

In [2129]:
class Chebyshev:
    """
    Chebyshev(a, b, n, func)
    Given a function func, lower and upper limits of the interval [a,b],
    and maximum degree n, this class computes a Chebyshev approximation
    of the function.
    Method eval(x) yields the approximated function value.
    """
    def __init__(self, a, b, n, func):
        n = n + 1
        self.a = a
        self.b = b
        self.func = func

        bma = 0.5 * (b - a)
        bpa = 0.5 * (b + a)
        f = [func(math.cos(math.pi * (k + 0.5) / n) * bma + bpa) for k in range(n)]
        self.roots = f

        self.x = [math.cos(math.pi * (k + 0.5) / n) * bma + bpa for k in range(n)]

        fac = 2.0 / n
        self.c = [fac * sum([f[k] * math.cos(math.pi * j * (k + 0.5) / n)
                  for k in range(n)]) for j in range(n)]

    def eval(self, x):
        a,b = self.a, self.b
        #assert(a <= x <= b)
        y = (2.0 * x - a - b) * (1.0 / (b - a))
        y2 = 2.0 * y
        (d, dd) = (self.c[-1], 0)             # Special case first step for efficiency
        for cj in self.c[-2:0:-1]:            # Clenshaw's recurrence
            (d, dd) = (y2 * d - dd + cj, d)
        return y * d - dd + 0.5 * self.c[0]   # Last step is different

In [2130]:
a = 3200232430
b = 10000003

print(a.bit_length())
print(b.bit_length())

LUT_BITS = 8
LUT_SIZE = 1 << LUT_BITS

lut = [
    (1 << (bits * 2)) //
    ((1 << (bits - 1)) + (i << (bits - 1 - LUT_BITS)))
    for i in range(LUT_SIZE)
]

32
24


In [2131]:
lut[0].bit_length()

66

We now need to compute `b.bit_length()` homomorphically

In [2100]:
"""
Requires log(bits) multiplications to simulate the XOR
Q: Is there an algorithm that can "mask" the first 1 and mask out all the rest?
"""
def homomorphic_bit_length(arr):
    n = len(arr)
    padded = np.concatenate([arr, np.zeros(n, dtype=int)])  # [arr | 000...0]
    
    step = 1
    while step < n:
        padded = np.bitwise_or(padded, np.roll(padded, step))
        step *= 2
        
    return padded

In [2101]:
np.array(list(np.binary_repr(b, width=bits)))[::-1]

array(['1', '1', '0', '0', '0', '0', '0', '1', '0', '1', '1', '0', '1',
       '0', '0', '1', '0', '0', '0', '1', '1', '0', '0', '1', '0', '0',
       '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0',
       '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0',
       '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0'],
      dtype='<U1')

In [2102]:
bit_length_fhe = homomorphic_bit_length(np.array(list(np.binary_repr(b, width=bits)), dtype=int)[::-1])

bit_length_fhe

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [2103]:
summed = np.array(bit_length_fhe)

for i in range(round(math.log2(bits))):
    summed = summed + np.roll(summed, -2**i)
    
print(summed[:bits])

final_s = np.array(summed)
mask_final = np.zeros(len(final_s))
mask_final[bits - 1] = 1
final_s = final_s * mask_final

mask = np.zeros(len(bit_length_fhe))
mask[bits-1] = -1/(bits / 2) #evito di fare 128-b con sto trick

summed = summed * mask

mask2 = np.zeros(len(bit_length_fhe))
mask2[bits-1] = 1

summed = summed + mask2

summedmask = np.array(summed)

summed = summed + np.roll(summed, -1)
summed = summed + np.roll(summed, -2)
summed = summed + np.roll(summed, -4)

summed = summed - np.roll(summedmask, -7)

summed = np.roll(summed, -bits + 7)

summed

[64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64 64
 63 62 61 60 59 58 57 56 55 54 53 52 51 50 49 48 47 46 45 44 43 42 41 40
 39 38 37 36 35 34 33 32 31 30 29 28 27 26 25 24]


array([0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])

Given the encrypted `b._bit_length()`, we now need to perform a `blind shift` as:

```
b_norm = b << s
```


Ideas:
1) Converto in binario s (occupa 7 bits a 128-bits), ruoto numero logaritmico e maschero SI/NO se il bit corrente di s è 1

In [2104]:
# This is the trigonometric polynomial that can be used for the bit decomposition of integers in [0, 225]

# Note: this has been normalized to work over [-1, 1] instead of [0, 225] to save one multiplication

def mod_reduction_norm(bit_position):
    period = 2**bit_position
    
    assert period % 2 == 0 and period >= 2
    P = period
    seq = np.array([0]*(P//2) + [1]*(P//2), dtype=float)

    c = np.fft.fft(seq) / P

    def f(x):
        x *= (bits/2)
        x += bits/2
        s = c[0].real
        for k in range(1, P//2):
            ck = c[k]
            ang = 2 * math.pi * k * x / P
            s += 2 * (ck.real * math.cos(ang) - ck.imag * math.sin(ang))

        s += c[P//2].real * math.cos(math.pi * x)
        return s
    
    return f

In [2105]:
"""
X
s_bin: (Level: 11) [ 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ]
b_norm: (Level: 19) [ 0 0 0 0 0 0 0 0 0.999999 0.999999 0 0 0 0 0 1 0 0.999999 0.999998 0 1 0 0 1 0 0 0 0.999999 0.999998 0 0 0.999999 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ]
X
b_norm_rot: (Level: 11) [ 1 0 0 0 0.999999 0.999999 0 0 0.999999 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ]
idx (decimal): (Level: 13) [ 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 49 ]
13
X
b_norm: 2560000768
x: 7209912313
X
X
X
X
X
term1: 18457381058492656384
term2: 18436107088926446847
X
X
X
X
X
x1: 12501737326250278911
x2: 7205754844
X
X
X
X
X
term1: 18446737934659720192
term2: 18446750212759383039
X
X
X
X
X
x1: 12501747621287669516
x2: 7205757241
X
X
X
X
X
term1: 18446744070981561088
term2: 18446744076437542143
X
X
X
X
X
x1: 12501747619300638720
x2: 7205757241
X
X
X
X
X
""".count("X")

38

In [2106]:
38*9

342

In [2107]:
summed[:bits]

array([0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ])

In [2108]:
p1 = Chebyshev(-1, 1, 247, mod_reduction_norm(1))           # Period 2 {0, 1}
p2 = Chebyshev(-1, 1, 247, mod_reduction_norm(2))           # Period 4 {0, 0, 1, 1}
p3 = Chebyshev(-1, 1, 247, mod_reduction_norm(3))           # Period 8 {0, 0, 0, 0, 1, 1, 1, 1}
p4 = Chebyshev(-1, 1, 247, mod_reduction_norm(4))           # Period 16 
p5 = Chebyshev(-1, 1, 247, mod_reduction_norm(5))           # Period 32
p6 = Chebyshev(-1, 1, 247, mod_reduction_norm(6))           # Period 64
p7 = Chebyshev(-1, 1, 247, mod_reduction_norm(7))           # Period 128

In [2109]:
with open("p1-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p1.c).replace("[", "").replace("]", ""))
with open("p2-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p2.c).replace("[", "").replace("]", ""))
with open("p3-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p3.c).replace("[", "").replace("]", ""))
with open("p4-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p4.c).replace("[", "").replace("]", ""))
with open("p5-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p5.c).replace("[", "").replace("]", ""))
with open("p6-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p6.c).replace("[", "").replace("]", ""))
with open("p7-norm-247-LUT-DIVISION.txt", "w") as output:
    output.write(str(p7.c).replace("[", "").replace("]", ""))

In [2110]:
summed[0] = p1.eval(summed[0])
summed[1] = p2.eval(summed[1])
summed[2] = p3.eval(summed[2])
summed[3] = p4.eval(summed[3])
summed[4] = p5.eval(summed[4])
summed[5] = p6.eval(summed[5])
summed[6] = p7.eval(summed[6])

In [2111]:
p1.c

[0.9438005833206418,
 2.2854710267287553e-16,
 -0.05731470550033381,
 -7.830187738110998e-17,
 -0.060524465858539545,
 1.0457864344849782e-15,
 -0.06539497188366611,
 -1.0583850812937707e-15,
 -0.07112661926364376,
 -1.2710103536693404e-16,
 -0.07646901457855593,
 2.71584151677686e-16,
 -0.07966941972110018,
 2.3660452578409235e-16,
 -0.07853271988987742,
 8.88317657571741e-17,
 -0.07069389352045881,
 -1.6657956917302278e-16,
 -0.05419942702351923,
 -2.6866975038806423e-16,
 -0.028428067346289786,
 -7.07157068563723e-18,
 0.004781666570991661,
 -1.5332851743706785e-16,
 0.04019661915289419,
 -6.546393522607945e-16,
 0.06914558041248904,
 -1.152915532732305e-15,
 0.08119893412041736,
 9.908840062136289e-16,
 0.06804956728965385,
 3.2205521779090047e-16,
 0.02894560415663779,
 -1.9588714481444642e-16,
 -0.02477900716441242,
 4.823864298393542e-16,
 -0.07008911014810426,
 -6.457565057888446e-16,
 -0.08103830932661996,
 -7.327837088373586e-16,
 -0.04504595875611227,
 2.845766534374024e-16,

In [2112]:
summed[0]

4.6629367034256575e-15

In [2113]:
s_bin = np.vectorize(round)(summed)
s_bin[:bits]

int(''.join(s_bin[:bits][::-1].astype(int).astype(str)), 2)

40

In [2114]:
s_bin

array([0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# SIAMO QUA

A questo punto credo che abbiamo fatto (in 128 bits) 14 moltiplicazioni... quindi si bootstrappa

In [2115]:
b_norm = np.array(list(np.binary_repr(b, width=bits)), dtype=int)[::-1]

b_norm = np.concatenate([b_norm, np.zeros(bits, dtype=int)])  # [arr | 000...0]

b_norm

array([1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [2116]:
for i in range(7):
    mask = np.zeros(len(b_norm))
    mask[i]  = s_bin[i]
    mask = np.roll(mask, -i) #Rimetto in pos 0
    
    for j in range(round(math.log2(bits))):
        mask = mask + np.roll(mask, 2**j)
    
    b_norm = b_norm * (1 - mask) + np.roll(b_norm, 2**i) * mask
    
b_norm

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 1., 1.,
       0., 1., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [2117]:
int(''.join(b_norm_rot[:128][::-1].astype(int).astype(str)), 2)

49

In [2118]:
#b_norm_rot = np.concatenate([b_norm_rot, np.zeros(bits, dtype=int)])  # [arr | 000...0]
b_norm_rot = np.roll(b_norm, -(bits - 1 - LUT_BITS))

int(''.join(b_norm_rot[:128][::-1].astype(int).astype(str)), 2)


59846413591472423246442500690590826801

In [2119]:
b_norm_rot

array([1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.,
       0., 1., 0., 1., 1., 0., 1., 0., 0.])

In [2120]:
mask = np.zeros(len(b_norm_rot))
mask[:LUT_BITS] = 1
print(b_norm_rot)
b_norm_rot = b_norm_rot * mask
idx = int(''.join(b_norm_rot[::-1].astype(int).astype(str)), 2)
print("idx", idx)

[1. 0. 0. 0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0.
 1. 0. 1. 1. 0. 1. 0. 0.]
idx 49


In [2121]:
idx

49

### Todo: map the LUT as a polynomial

In [2122]:
def get_bit(n, i):
    return (n >> i) & 1

In [2123]:
def polylut(bit = 0):
    N = LUT_SIZE
    seq = np.array([get_bit(lut[i], bit) for i in range(N)], dtype=float)
    c = np.fft.fft(seq)  

    def f(x):
        s = 0.0
        for k in range(N):
            ang = 2 * math.pi * k * x / N
            s += c[k].real * math.cos(ang) - c[k].imag * math.sin(ang)
        return s / N

    return f

In [2124]:
len(lut)

256

In [2125]:
# SKIPPALO SE GIA AVVIATO, PESANTE 

if True:
    polyluts = []

    for bit in range(bits + 2):
        f = polylut(bit)

        c = Chebyshev(0, LUT_SIZE, 851, f)

        polyluts.append(c)

In [2132]:
for i in range(bits + 2):
    with open("LUT-DIVISION-{}-bits-{}.txt".format(bits, i), "w") as output:
        output.write(str(polyluts[i].c).replace("[", "").replace("]", ""))

In [2024]:
LUT_SIZE

256

---

### Making the LUT polynomial

In [2091]:
binx = []

for i in range(bits + 2):
    if i >= len(polyluts):
        binx.append(0)
    else:
        binx.append(polyluts[i].eval(idx))

In [2092]:
binx

/Users/narger/Library/Python/3.9/lib/python/site-packages/IPython/core/displayhook.py:275: UserWarning: Output cache limit (currently 1000 entries) hit.
Flushing oldest 200 entries.
  warn('Output cache limit (currently {sz} entries) hit.\n'


[1.0000000004398648,
 -5.165328331724339e-09,
 1.1433360924684166e-09,
 0.999999997987769,
 0.9999999981469878,
 1.0000000005707643,
 1.0000000004843306,
 1.0000000045264126,
 1.0000000007838148,
 0.9999999946113931,
 0.9999999989371781,
 -1.2833379914312104e-09,
 -1.575669039688421e-09,
 1.4938356107663253e-09,
 -4.436118139494738e-10,
 0.9999999971793521,
 -4.251775820307557e-09,
 0.9999999997009853,
 0.9999999995873385,
 1.0000000042025343,
 0.9999999941724574,
 0.9999999982984248,
 2.269800181142756e-09,
 0.9999999965483044,
 0.9999999993924356,
 -1.3316145963671033e-09,
 1.00000000057253,
 0.9999999991970072,
 -1.8906601284562896e-09,
 0.999999998818638,
 3.349842625510746e-11,
 1.0000000339161184,
 0.9999999997959031,
 2.0400218320171248e-10]

In [2093]:
polydec = ([round(x) for x in list(binx)][::-1])

In [2094]:
n = int("".join(map(str, polydec)), 2)

# We check that the polynomial LUT and the actual LUT are equal
n == lut[idx]

True

In [2029]:
lut[idx]

7209912313

In [2030]:
b_norm

array([0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0.,
       1., 1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [2031]:
n

7209912313

In [2032]:
lut[idx]

7209912313

In [2033]:
test = [0, 0,0,0,0,0,0, 0, 1]
int("".join(map(str, test[::-1])), 2)

256

In [2048]:
x = n
b_norm_int = int("".join(map(str, np.vectorize(round)(b_norm)[::-1])), 2)

print("b_norm:", b_norm_int)
print("x:", x)

for _ in range(3):
    term = b_norm_int * x        # 64 bit
    print("term1", term)
    term = 2**(32*2 + 1) - term        # 65 bit
    print("term2", term)
    term_high = term >> 32       # 33 bit (parte alta)
    
    print(x.bit_length())
    print(term_high.bit_length())

    x = x * term_high  
    print("x1", x)
    x = x >> 32                  # torna a 64 bit
    print("x2", x)

result = (a * x)
print("result before blind:", result)

result = result >> (bits)

print("result before blind:", result)

result = result << 8

print("result before blind:", result)
result = result >> (bits)     # Blind rotation

print("result before blind:", result)

print(result)
print(a // b)

b_norm: 2560000768
x: 7209912313
term1 18457381058492656384
term2 18436107088926446848
33
32
x1 30948481399959830527
x2 7205754844
term1 18446737934659720192
term2 18446750212759383040
33
33
x1 30948491694997253900
x2 7205757241
term1 18446744070981561088
term2 18446744076437542144
33
33
x1 30948491693010190336
x2 7205757241
result before blind: 23060098005355525630
result before blind: 5369097461
result before blind: 1374488950016
result before blind: 320
320
320


In [2052]:
12 << 8

3072

In [ ]:
il vero è 24, io ho 8

In [ ]:
32 e 24

3

In [2043]:
32+32-24

40

In [2042]:
final_s

array([ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0., 24.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])

In [2035]:
prova = np.array(list(np.binary_repr(5369097461, width=bits*2)), dtype=int)[::-1]
prova

array([1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [2036]:
provas = np.array(list(np.binary_repr(round(final_s.sum()), width=bits*2)), dtype=int)[::-1]
provas

array([0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [2037]:
for i in range(7):
    mask = np.zeros(len(prova))
    mask[i]  = provas[i]
    print(mask[i])
    mask = np.roll(mask, -i) #Rimetto in pos 0
    
    for j in range(round(math.log2(bits * 2))):
        mask = mask + np.roll(mask, 2**j)
    
    prova = prova * (1 - mask) + np.roll(prova, -2**i) * mask
    
prova = prova[:32]

0.0
0.0
0.0
1.0
1.0
0.0
0.0


In [ ]:
>> 32 (32 - 8

In [2038]:
final_s.sum()

24.0

In [2039]:
int("".join(map(str, np.vectorize(round)(prova)[::-1])), 2)

320

In [2040]:
result = 4613353934297625933 >> round(final_s.sum())
result

274977322476

In [1932]:
2**8

256

In [1665]:
term.bit_length()

46

In [1666]:
fhebnorm = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.999999, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [1485]:
b_norm - fhebnorm

array([0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 1.e-06, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00,
       0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00, 0.e+00])